# 03. Business Feature Engineering & RFM Customer Segmentation
**GitGuide Analytics**


## 1. Setup & Environment Imports


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid')


## 2. Constructing Time & Volume Ratio Features
Normalize transaction metrics by tenure and order volume.


In [ ]:
df = pd.read_csv('../data/raw/customer_activity.csv')
months = df['days_as_customer'] / 30.0

df['transactions_per_month'] = np.round(df['total_transactions'] / months, 2)
df['avg_spend_per_transaction'] = np.round(df['total_spent'] / df['total_transactions'], 2)
df['lifetime_value_per_month'] = np.round(df['total_spent'] / months, 2)

df[['customer_id', 'transactions_per_month', 'avg_spend_per_transaction', 'lifetime_value_per_month']].head()


## 3. Binning Strategies (pd.cut & pd.qcut)
Apply fixed-range binning for engagement and quantile binning for spend tiers.


In [ ]:
df['engagement_tier'] = pd.cut(df['transactions_per_month'], bins=[0, 2, 10, np.inf], labels=['low', 'medium', 'high'])
df['spend_tier'] = pd.qcut(df['total_spent'], q=4, labels=['tier_1', 'tier_2', 'tier_3', 'tier_4'])

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='engagement_tier', palette='viridis').set(title='Engagement Tiers (pd.cut)')
plt.subplot(1, 2, 2)
sns.countplot(data=df, x='spend_tier', palette='magma').set(title='Spend Quantiles (pd.qcut)')
plt.tight_layout()
plt.show()


## 4. Composite RFM Customer Health Score
Calculate Recency, Frequency, and Monetary scores.


In [ ]:
df['r_score'] = pd.qcut(df['days_since_last_purchase'], q=5, labels=[5,4,3,2,1]).astype(int)
df['f_score'] = pd.qcut(df['total_transactions'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
df['m_score'] = pd.qcut(df['total_spent'], q=5, labels=[1,2,3,4,5]).astype(int)
df['rfm_score'] = df['r_score'] + df['f_score'] + df['m_score']

plt.figure(figsize=(8, 4))
sns.histplot(df['rfm_score'], discrete=True, color='teal', edgecolor='black')
plt.title('Composite RFM Score Distribution (Range: 3 - 15)')
plt.xlabel('RFM Score')
plt.ylabel('Customer Count')
plt.show()
